# LeetCode 1937 — Maximum Number of Points with Cost bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-points-with-cost/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Matrix

Skeleton LeetCode:

```python
class Solution:
    def maxPoints(self, points: List[List[int]]) -> int:
        ...
```

Lộ trình: **thử mọi cách chọn cột** `O(nᵐ · m)` → **DP theo hàng, mỗi ô xét mọi cột hàng trên** `O(m·n²)` → **DP + hai lượt quét trái/phải (max tiền tố có trừ khoảng cách)** `O(m·n)` (lời giải nộp) → **tách `|c₁ − c₂|` thành `dp[j] + j` và `dp[j] − j`** (cùng ý tưởng, nhìn dưới góc đại số).

## 1. Mục tiêu học

1. Xây DP theo hàng: `dp[r][c]` = điểm lớn nhất khi chọn ô `(r, c)` ở hàng `r`.
2. Nhận ra nút thắt `max_j (dp[j] − |j − c|)` và khử nó bằng **hai lượt quét** trái → phải và phải → trái.
3. Hiểu kỹ thuật **tách trị tuyệt đối**: `−|j − c| = −(c − j)` khi `j ≤ c`, `= −(j − c)` khi `j ≥ c` ⇒ hai bài "max tiền tố".
4. Dùng **bộ nhớ cuộn** (một mảng `dp` cho hàng trước) vì chỉ phụ thuộc hàng liền trên.
5. Kiểm chứng bằng vét cạn trên ma trận nhỏ ngẫu nhiên có seed; thấy vì sao tham lam theo hàng sai.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `points` kích thước `m × n`. Bắt đầu với 0 điểm; **mỗi hàng chọn đúng một ô**, chọn ô `(r, c)` được cộng `points[r][c]`.

Nhưng với hai hàng liền nhau `r` và `r + 1`, nếu chọn ô `(r, c₁)` và `(r + 1, c₂)` thì bị **trừ** `|c₁ − c₂|`.

Trả về số điểm **lớn nhất** đạt được.

**Ví dụ 1:** `points = [[1,2,3],[1,5,1],[3,1,1]]` → `9`. Chọn `(0,2)`, `(1,1)`, `(2,0)`: `3 + 5 + 3 − |2−1| − |1−0| = 11 − 2 = 9`.

**Ví dụ 2:** `points = [[1,5],[2,3],[4,2]]` → `11`. Chọn `(0,1)`, `(1,1)`, `(2,0)`: `5 + 3 + 4 − 0 − 1 = 11`.

**Ràng buộc:** `1 ≤ m, n ≤ 10⁵`, `1 ≤ m·n ≤ 10⁵`, `0 ≤ points[r][c] ≤ 10⁵`.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [([[1, 2, 3], [1, 5, 1], [3, 1, 1]], 9), ([[1, 5], [2, 3], [4, 2]], 11)]
for g, want in examples:
    print(g, "-> kỳ vọng", want)

def score(points, cols):
    return sum(points[r][c] for r, c in enumerate(cols)) - sum(abs(a - b) for a, b in zip(cols, cols[1:]))

print("điểm của lựa chọn (2,1,0) ở Ví dụ 1:", score(examples[0][0], (2, 1, 0)))

[[1, 2, 3], [1, 5, 1], [3, 1, 1]] -> kỳ vọng 9
[[1, 5], [2, 3], [4, 2]] -> kỳ vọng 11
điểm của lựa chọn (2,1,0) ở Ví dụ 1: 9


## 3. Kiến thức nền

### 3.1 DP theo hàng và bộ nhớ cuộn

Hàng `r` chỉ phụ thuộc hàng `r − 1` ⇒ giữ một list `dp` độ dài `n` cho hàng trước, tính `ndp` cho hàng hiện tại rồi gán `dp = ndp`.

In [2]:
dp = [1, 2, 3]
ndp = [x * 10 for x in dp]
dp = ndp
print(dp)

[10, 20, 30]


### 3.2 Max tiền tố

`pre[c] = max(a[0], …, a[c])` tính một lượt: `pre[c] = max(pre[c−1], a[c])`. Ở bài này có thêm "trừ 1 mỗi bước": `left[c] = max(left[c−1] − 1, a[c])`.

In [3]:
a = [5, 1, 7, 2]
left = a[:]
for c in range(1, len(a)):
    left[c] = max(left[c - 1] - 1, a[c])
print("left (max_{j<=c} a[j] - (c - j)):", left)
print("kiểm trực tiếp:", [max(a[j] - (c - j) for j in range(c + 1)) for c in range(len(a))])

left (max_{j<=c} a[j] - (c - j)): [5, 4, 7, 6]
kiểm trực tiếp: [5, 4, 7, 6]


### 3.3 Tách trị tuyệt đối

`max_j (a[j] − |j − c|) = max( max_{j ≤ c} (a[j] − (c − j)), max_{j ≥ c} (a[j] − (j − c)) )`. Mỗi vế là một "max tiền tố/hậu tố có trừ bước".

In [4]:
a = [5, 1, 7, 2]
right = a[:]
for c in range(len(a) - 2, -1, -1):
    right[c] = max(right[c + 1] - 1, a[c])
best = [max(l, r) for l, r in zip(left, right)]
print("best:", best, "| trực tiếp:", [max(a[j] - abs(j - c) for j in range(len(a))) for c in range(len(a))])

best: [5, 6, 7, 6] | trực tiếp: [5, 6, 7, 6]


### 3.4 `itertools.product` để liệt kê mọi cách chọn cột

In [5]:
print(list(product(range(2), repeat=3)))

[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1, tính tay DP:

- Hàng 0: `dp = [1, 2, 3]`.
- Hàng 1: với cột `c`, lấy `max_j (dp[j] − |j − c|)` rồi cộng `points[1][c]`:
  - `c = 0`: `max(1−0, 2−1, 3−2) = 1` ⇒ `1 + 1 = 2`
  - `c = 1`: `max(1−1, 2−0, 3−1) = 2` ⇒ `2 + 5 = 7`
  - `c = 2`: `max(1−2, 2−1, 3−0) = 3` ⇒ `3 + 1 = 4`
  - `dp = [2, 7, 4]`
- Hàng 2: `c = 0`: `max(2, 7−1, 4−2) = 6` ⇒ `6 + 3 = 9`; `c = 1`: `max(1, 7, 3) = 7` ⇒ `8`; `c = 2`: `max(0, 6, 4) = 6` ⇒ `7`. `dp = [9, 8, 7]`.

Đáp án `max(dp) = 9`. ✔ Phần tốn kém là "`max_j (dp[j] − |j − c|)`" — làm ngây thơ mất `O(n)` cho **mỗi** `c`.

## 5. Cách 1 — Brute Force / baseline: thử mọi cách chọn cột

In [6]:
def max_points_brute(points):
    m, n = len(points), len(points[0])
    return max(score(points, cols) for cols in product(range(n), repeat=m))

for g, want in examples:
    print(max_points_brute(g), "| kỳ vọng", want)

9 | kỳ vọng 9
11 | kỳ vọng 11


**Độ phức tạp:** `nᵐ` cách chọn × `O(m)` ⇒ hàm mũ. **Nút thắt:** hai cách chọn trùng cột ở hàng `r` thì phần còn lại (hàng `r+1..`) giống hệt nhau ⇒ chỉ cần nhớ **giá trị tốt nhất kết thúc tại mỗi cột** ⇒ DP.

## 6. Ý tưởng thuật toán / bước đột phá

**DP:** `dp_r[c] = points[r][c] + max_j (dp_{r−1}[j] − |j − c|)`; đáp án `max_c dp_{m−1}[c]`.

**Nút thắt:** `max_j (dp[j] − |j − c|)` cho mọi `c` — ngây thơ `O(n²)`.

**Insight (tách trị tuyệt đối):**
- `L[c] = max_{j ≤ c} (dp[j] − (c − j))` ⇒ truy hồi `L[c] = max(L[c−1] − 1, dp[c])` (đi sang phải một cột thì mọi ứng viên bên trái bị trừ thêm 1).
- `R[c] = max_{j ≥ c} (dp[j] − (j − c))` ⇒ `R[c] = max(R[c+1] − 1, dp[c])`.
- `max_j (dp[j] − |j − c|) = max(L[c], R[c])`.

**Quy tắc:** mỗi hàng: quét trái → phải tính `L`, quét phải → trái tính `R`, rồi `ndp[c] = points[r][c] + max(L[c], R[c])`.

**Invariant:** sau khi xử lý hàng `r`, `dp[c]` là điểm tốt nhất của mọi cách chọn ở các hàng `0..r` mà hàng `r` chọn cột `c`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP `O(m·n²)`

In [7]:
def max_points_quadratic(points):
    dp = points[0][:]
    for row in points[1:]:
        n = len(row)
        dp = [row[c] + max(dp[j] - abs(j - c) for j in range(n)) for c in range(n)]
    return max(dp)

for g, want in examples:
    print(max_points_quadratic(g), want)

9 9
11 11


**Độ phức tạp:** `O(m·n²)`; với `m = 1` không sao, nhưng `m = 2, n = 5·10⁴` đã là `2.5·10⁹`. Nút thắt: lượt `max` bên trong.

### 7.2 Cách 3 — DP + hai lượt quét (`O(m·n)`) — lời giải chuẩn

In [8]:
def max_points_scan_demo(points):
    dp = points[0][:]
    print("hàng 0:", dp)
    for r, row in enumerate(points[1:], start=1):
        n = len(row)
        L = dp[:]
        for c in range(1, n):
            L[c] = max(L[c - 1] - 1, dp[c])
        R = dp[:]
        for c in range(n - 2, -1, -1):
            R[c] = max(R[c + 1] - 1, dp[c])
        dp = [row[c] + max(L[c], R[c]) for c in range(n)]
        print(f"hàng {r}: L={L}, R={R} -> dp={dp}")
    return max(dp)

print("=>", max_points_scan_demo(examples[0][0]))

hàng 0: [1, 2, 3]
hàng 1: L=[1, 2, 3], R=[1, 2, 3] -> dp=[2, 7, 4]
hàng 2: L=[2, 7, 6], R=[6, 7, 4] -> dp=[9, 8, 7]
=> 9


In [9]:
class SolutionScan:
    def maxPoints(self, points: List[List[int]]) -> int:
        dp = points[0][:]
        n = len(dp)
        for row in points[1:]:
            L = dp[:]
            for c in range(1, n):
                if L[c - 1] - 1 > L[c]:
                    L[c] = L[c - 1] - 1
            R = dp[:]
            for c in range(n - 2, -1, -1):
                if R[c + 1] - 1 > R[c]:
                    R[c] = R[c + 1] - 1
            dp = [row[c] + (L[c] if L[c] > R[c] else R[c]) for c in range(n)]
        return max(dp)

for g, want in examples:
    print(SolutionScan().maxPoints(g), want)

9 9
11 11


**Độ phức tạp:** mỗi hàng 3 lượt `O(n)` ⇒ `O(m·n) ≤ 3·10⁵` phép cơ bản; bộ nhớ `O(n)`.

### 7.3 Cách 4 — Góc nhìn đại số: `dp[j] + j` và `dp[j] − j`

`dp[j] − (c − j) = (dp[j] + j) − c` với `j ≤ c` ⇒ `L[c] = (max_{j≤c} (dp[j] + j)) − c` — **max tiền tố thuần** của mảng `dp[j] + j`. Tương tự `R[c] = (max_{j≥c} (dp[j] − j)) + c`. Viết bằng `itertools.accumulate(…, max)`.

In [10]:
from itertools import accumulate

class SolutionAccumulate:
    def maxPoints(self, points: List[List[int]]) -> int:
        dp = points[0][:]
        n = len(dp)
        for row in points[1:]:
            left = list(accumulate((dp[j] + j for j in range(n)), max))
            right = list(accumulate((dp[j] - j for j in range(n - 1, -1, -1)), max))[::-1]
            dp = [row[c] + max(left[c] - c, right[c] + c) for c in range(n)]
        return max(dp)

for g, want in examples:
    print(SolutionAccumulate().maxPoints(g), want)

9 9
11 11


Hai cách 3 và 4 là **cùng một thuật toán**; cách 4 làm rõ vì sao truy hồi "trừ 1 mỗi bước" đúng: nó chính là max tiền tố của `dp[j] + j` sau khi trừ `c`.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 2 `[[1,5],[2,3],[4,2]]`:

| hàng | `dp` trước | `L` | `R` | `ndp = row + max(L, R)` |
|---|---|---|---|---|
| 0 | — | — | — | `[1, 5]` |
| 1 | `[1, 5]` | `[1, 5]` | `[4, 5]` | `[2+4, 3+5] = [6, 8]` |
| 2 | `[6, 8]` | `[6, 8]` | `[7, 8]` | `[4+7, 2+8] = [11, 10]` |

Đáp án `max = 11`. ✔

In [11]:
max_points_scan_demo(examples[1][0])

hàng 0: [1, 5]
hàng 1: L=[1, 5], R=[4, 5] -> dp=[6, 8]
hàng 2: L=[6, 8], R=[7, 8] -> dp=[11, 10]


11

Truy vết **cột được chọn** (lưu nguồn của `L`/`R`):

In [12]:
def best_columns(points):
    m, n = len(points), len(points[0])
    dp = points[0][:]
    parent = []
    for row in points[1:]:
        L = dp[:]; Lsrc = list(range(n))
        for c in range(1, n):
            if L[c - 1] - 1 > L[c]:
                L[c] = L[c - 1] - 1; Lsrc[c] = Lsrc[c - 1]
        R = dp[:]; Rsrc = list(range(n))
        for c in range(n - 2, -1, -1):
            if R[c + 1] - 1 > R[c]:
                R[c] = R[c + 1] - 1; Rsrc[c] = Rsrc[c + 1]
        src = [Lsrc[c] if L[c] >= R[c] else Rsrc[c] for c in range(n)]
        dp = [row[c] + max(L[c], R[c]) for c in range(n)]
        parent.append(src)
    c = max(range(n), key=lambda k: dp[k])
    cols = [c]
    for src in reversed(parent):
        c = src[c]; cols.append(c)
    return max(dp), cols[::-1]

for g, want in examples:
    v, cols = best_columns(g)
    print("điểm", v, "cột", cols, "| kiểm lại:", score(g, cols))

điểm 9 cột [1, 1, 0] | kiểm lại: 9
điểm 11 cột [1, 1, 0] | kiểm lại: 11


## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def maxPoints(self, points: List[List[int]]) -> int:
        n = len(points[0])
        dp = points[0][:]
        for row in points[1:]:
            left = dp[:]                       # left[c] = max_{j<=c} dp[j] - (c - j)
            for c in range(1, n):
                left[c] = max(left[c], left[c - 1] - 1)
            right = dp[:]                      # right[c] = max_{j>=c} dp[j] - (j - c)
            for c in range(n - 2, -1, -1):
                right[c] = max(right[c], right[c + 1] - 1)
            dp = [row[c] + max(left[c], right[c]) for c in range(n)]
        return max(dp)

for g, want in examples:
    assert Solution().maxPoints(g) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**DP đúng.** Mọi cách chọn hàng `0..r` kết thúc ở cột `c` có điểm = (điểm tốt nhất có thể của hàng `0..r−1` kết thúc ở một cột `j`) `− |j − c| + points[r][c]`. Lấy max theo `j` ⇒ `dp_r[c]`. Đáp án là max theo cột ở hàng cuối.

**Lượt quét trái đúng.** Chứng minh `left[c] = max_{j ≤ c} (dp[j] − (c − j))` bằng quy nạp: `c = 0` hiển nhiên. Với `c ≥ 1`: tập `j ≤ c` = `{c} ∪ {j ≤ c − 1}`; ứng viên `j = c` cho `dp[c]`; mọi `j ≤ c − 1` cho `dp[j] − (c − 1 − j) − 1`, max của chúng là `left[c−1] − 1`. Lượt phải đối xứng. `max(left[c], right[c])` phủ mọi `j` (`j = c` nằm ở cả hai) ⇒ bằng `max_j (dp[j] − |j − c|)`.

## 11. Phân tích Time / Space

Gọi `m` hàng, `n` cột, `m·n ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(nᵐ · m)` | `O(m)` |
| 2. DP `O(n²)` mỗi hàng | `O(m · n²)` | `O(n)` |
| 3. DP + 2 lượt quét | `O(m · n)` | `O(n)` |
| 4. DP + accumulate | `O(m · n)` | `O(n)` |

Đáp án có thể tới `m · 10⁵ ≤ 10¹⁰` — vượt `int32` (Python không lo).

In [14]:
rng = random.Random(1937)
shapes = [(1, 10**5), (2, 5 * 10**4), (316, 316), (10**5, 1)]
for m, n in shapes:
    g = [[rng.randint(0, 10**5) for _ in range(n)] for _ in range(m)]
    res = []
    for name, sol in [("Quét/Nộp", Solution()), ("Accumulate", SolutionAccumulate())]:
        t = time.perf_counter(); r = sol.maxPoints(g); res.append(r)
        print(f"{m}x{n} {name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1
g = [[rng.randint(0, 10**5) for _ in range(800)] for _ in range(2)]
t = time.perf_counter(); max_points_quadratic(g); dt = time.perf_counter() - t
print(f"Cách 2 với 2x800: {dt:.2f}s -> 2x5·10^4 ước tính ≈ {dt * (5e4 / 800) ** 2:.0f}s")

1x100000 Quét/Nộp: 100000 trong 2 ms
1x100000 Accumulate: 100000 trong 1 ms
2x50000 Quét/Nộp: 199981 trong 10 ms
2x50000 Accumulate: 199981 trong 10 ms
316x316 Quét/Nộp: 31471047 trong 13 ms
316x316 Accumulate: 31471047 trong 17 ms


100000x1 Quét/Nộp: 4997424918 trong 23 ms


100000x1 Accumulate: 4997424918 trong 80 ms
Cách 2 với 2x800: 0.03s -> 2x5·10^4 ước tính ≈ 125s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [15]:
sols = [Solution(), SolutionScan(), SolutionAccumulate()]
edge = examples + [
    ([[7]], 7),
    ([[1, 9, 2]], 9),                    # một hàng: chọn max
    ([[5], [3], [8]], 16),               # một cột: không bị trừ
    ([[0, 0, 10], [10, 0, 0]], 18),      # đổi cột tốn 2
    ([[10, 0, 0, 0], [0, 0, 0, 10]], 17),
    ([[0, 0], [0, 0]], 0),
]
for g, want in edge:
    assert max_points_brute(g) == want, (g, max_points_brute(g))
    for s in sols:
        assert s.maxPoints(g) == want, (g, s)
    assert max_points_quadratic(g) == want
    print("OK", g, "->", want)

OK [[1, 2, 3], [1, 5, 1], [3, 1, 1]] -> 9
OK [[1, 5], [2, 3], [4, 2]] -> 11
OK [[7]] -> 7
OK [[1, 9, 2]] -> 9
OK [[5], [3], [8]] -> 16
OK [[0, 0, 10], [10, 0, 0]] -> 18
OK [[10, 0, 0, 0], [0, 0, 0, 10]] -> 17
OK [[0, 0], [0, 0]] -> 0


### 12.2 Ngẫu nhiên có seed: so với vét cạn (ma trận nhỏ) và DP `O(n²)` (vừa)

In [16]:
rng = random.Random(42)
for _ in range(500):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    hi = rng.choice([3, 10, 10**5])
    g = [[rng.randint(0, hi) for _ in range(n)] for _ in range(m)]
    want = max_points_brute(g)
    for s in sols:
        assert s.maxPoints(g) == want, g
    v, cols = best_columns(g)
    assert v == want == score(g, cols)
for _ in range(30):
    m, n = rng.randint(1, 30), rng.randint(1, 60)
    g = [[rng.randint(0, 100) for _ in range(n)] for _ in range(m)]
    assert len({s.maxPoints(g) for s in sols} | {max_points_quadratic(g)}) == 1
print("500 ma trận nhỏ (vét cạn) + 30 ma trận vừa (DP O(n²)): khớp")

500 ma trận nhỏ (vét cạn) + 30 ma trận vừa (DP O(n²)): khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam: mỗi hàng chọn ô lớn nhất

In [17]:
def wrong_greedy(points):
    cols = [max(range(len(row)), key=lambda c: row[c]) for row in points]
    return score(points, cols)

for g in ([[4, 7, 4, 8], [5, 5, 4, 5], [6, 5, 2, 7]], [[0, 0, 10], [10, 0, 0]], examples[0][0]):
    print(g, "tham lam:", wrong_greedy(g), "| đúng:", Solution().maxPoints(g))

[[4, 7, 4, 8], [5, 5, 4, 5], [6, 5, 2, 7]] tham lam: 14 | đúng: 20
[[0, 0, 10], [10, 0, 0]] tham lam: 18 | đúng: 18
[[1, 2, 3], [1, 5, 1], [3, 1, 1]] tham lam: 9 | đúng: 9


**Vì sao sai:** chọn max từng hàng bỏ qua chi phí di chuyển. Với `[[4,7,4,8],[5,5,4,5],[6,5,2,7]]` (tìm bằng tìm kiếm ngẫu nhiên, đối chiếu vét cạn): tham lam chọn cột `3, 0, 3` — nhảy qua lại tốn `3 + 3` — chỉ được 14, còn tối ưu 20 (ví dụ cột `3, 3, 3`: `8 + 5 + 7`). Hai ví dụ còn lại tham lam **tình cờ** đúng — một lần nữa, test theo ví dụ đề là không đủ.

### Lỗi 2 — Chỉ quét một chiều (quên các cột bên phải)

In [18]:
def wrong_one_side(points):
    dp = points[0][:]; n = len(dp)
    for row in points[1:]:
        left = dp[:]
        for c in range(1, n):
            left[c] = max(left[c], left[c - 1] - 1)
        dp = [row[c] + left[c] for c in range(n)]          # SAI: thiếu ứng viên j > c
    return max(dp)

for g, want in examples:
    print(g, "sai:", wrong_one_side(g), "| đúng:", want)

[[1, 2, 3], [1, 5, 1], [3, 1, 1]] sai: 8 | đúng: 9
[[1, 5], [2, 3], [4, 2]] sai: 10 | đúng: 11


**Vì sao sai:** `left[c]` chỉ xét cột `j ≤ c` của hàng trên; Ví dụ 1 cần đi từ cột 2 sang cột 1 (từ **phải** sang trái). **Sửa:** thêm lượt quét phải → trái.

### Lỗi 3 — Quên "trừ 1 mỗi bước" khi quét (thành max tiền tố thường)

In [19]:
def wrong_no_penalty(points):
    dp = points[0][:]; n = len(dp)
    for row in points[1:]:
        left = dp[:]
        for c in range(1, n): left[c] = max(left[c], left[c - 1])        # SAI: thiếu -1
        right = dp[:]
        for c in range(n - 2, -1, -1): right[c] = max(right[c], right[c + 1])
        dp = [row[c] + max(left[c], right[c]) for c in range(n)]
    return max(dp)

for g, want in examples:
    print(g, "sai:", wrong_no_penalty(g), "| đúng:", want)

[[1, 2, 3], [1, 5, 1], [3, 1, 1]] sai: 11 | đúng: 9
[[1, 5], [2, 3], [4, 2]] sai: 12 | đúng: 11


**Vì sao sai:** không trừ khoảng cách ⇒ coi như đổi cột miễn phí ⇒ đáp án quá lớn (thực chất là tổng max từng hàng).

### Lỗi 4 — DP `O(m·n²)` khi `m` nhỏ, `n` lớn

In [20]:
g = [[rng.randint(0, 10**5) for _ in range(1500)] for _ in range(2)]
t = time.perf_counter(); r1 = max_points_quadratic(g); t1 = time.perf_counter() - t
t = time.perf_counter(); r2 = Solution().maxPoints(g); t2 = time.perf_counter() - t
print(f"2x1500: O(n²) {t1:.2f}s vs quét {1000*t2:.1f} ms, cùng kết quả: {r1 == r2}")

2x1500: O(n²) 0.12s vs quét 0.3 ms, cùng kết quả: True


**Vì sao chưa tốt:** ràng buộc chỉ chặn `m·n ≤ 10⁵`, nên `n` có thể tới `5·10⁴` với `m = 2` ⇒ `O(n²)` mỗi hàng là TLE.

### Lỗi 5 — Không dùng giá trị cũ khi cập nhật (dùng chung một mảng cho `dp` và `left`)

In [21]:
def inplace_both(points):
    dp = points[0][:]; n = len(dp)
    for row in points[1:]:
        for c in range(1, n): dp[c] = max(dp[c], dp[c - 1] - 1)           # tại chỗ
        for c in range(n - 2, -1, -1): dp[c] = max(dp[c], dp[c + 1] - 1)   # tại chỗ
        dp = [row[c] + dp[c] for c in range(n)]
    return max(dp)

rng = random.Random(1)
print("tại chỗ có sai không?", any(inplace_both(g) != Solution().maxPoints(g)
      for g in ([[rng.randint(0, 20) for _ in range(6)] for _ in range(4)] for _ in range(500))))

tại chỗ có sai không? False


**Ghi chú (không phải lỗi):** quét **tại chỗ** cả hai chiều trên cùng một mảng vẫn **đúng** — sau lượt trái, `dp[c] = L[c]`; lượt phải lấy `max(L[c], L[c+1] − 1, …)`, mà `L[c+1] − 1` không bao giờ vượt `max(L[c], R[c])` thật. Kiểm bằng máy cho kết quả `False` (không có sai). Đây là một tối ưu bộ nhớ hợp lệ — nhưng cần lập luận, không nên làm theo cảm tính.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **DP lưới, chuyển từ hàng trên với chi phí `|c₁ − c₂|`** ⇒ tách trị tuyệt đối thành hai lượt quét (hoặc max tiền tố của `dp[j] ± j`).
- Mẫu tổng quát: `max_j (f(j) − w·|j − c|)` ⇒ hai lượt với bước trừ `w`.
- Chi phí **lồi** tổng quát `cost(j − c)` ⇒ cần kỹ thuật mạnh hơn (convex hull trick, SMAWK…).
- Bộ nhớ cuộn: chỉ giữ hàng trước.
- Ràng buộc kiểu `m·n ≤ 10⁵` ⇒ thuật toán phải tuyến tính theo `m·n`, không được `n²`.

## 15. Biến thể của bài

1. **Chi phí `k·|c₁ − c₂|`** ⇒ quét với bước trừ `k` (Bài tập 5).
2. **Chỉ được chuyển sang cột lệch ≤ 1** (LeetCode 931 *Minimum Falling Path Sum*) ⇒ DP 3 ứng viên.
3. **Không được chọn cùng cột hai hàng liền nhau** (LeetCode 1289) ⇒ giữ min nhỏ nhất và nhì.
4. **Chi phí `(c₁ − c₂)²`** ⇒ convex hull trick.
5. **Hỏi cả đường đi** ⇒ lưu nguồn của `L`/`R` (mục 8).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi cách chọn | `O(nᵐ m)` | `O(m)` | kiểm chứng |
| 2. DP `O(n²)` | max mọi cột hàng trên | `O(m n²)` | `O(n)` | `n` nhỏ |
| 3. DP + quét | `L`, `R` trừ 1 mỗi bước | `O(m n)` | `O(n)` | **nộp bài** |
| 4. DP + accumulate | max tiền tố `dp[j] ± j` | `O(m n)` | `O(n)` | viết gọn, thấy bản chất |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `[[3]]`, `[[1,2],[3,4]]`, `[[5,0],[0,5]]`?

**Bài 2 (tính tay):** Tính `L`, `R`, `dp` cho hàng thứ hai của `[[0,0,10],[10,0,0]]`.

**Bài 3 (điền TODO):** Viết `max_points_topdown(points)` bằng đệ quy có ghi nhớ trên `(r, c)` (chỉ dùng cho ma trận nhỏ) và đối chiếu.

**Bài 4 (tự viết lại):** Viết lại lời giải chỉ dùng **một** mảng phụ (tái sử dụng bộ nhớ giữa các hàng).

**Bài 5 (mini challenge):** Chi phí đổi cột là `k · |c₁ − c₂|` với `k ≥ 0` cho trước. Viết `max_points_k(points, k)` `O(m·n)`.

In [22]:
def max_points_topdown(points):
    # TODO
    pass

def max_points_one_buffer(points):
    # TODO
    pass

def max_points_k(points, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Một ô: chính nó | `[[1,2],[3,4]]`: `2 + 4 = 6` | `[[5,0],[0,5]]`: `5 + 5 − 1 = 9` |
| 2 | `dp = [0,0,10]` | `L = [0,0,10]`, `R = [8,9,10]` | `ndp = [18, 9, 10]` |
| 3 | `f(r, c)` = tốt nhất từ hàng `r` khi hàng `r` chọn `c` | `f(r,c) = p[r][c] + max_j (f(r+1, j) − |c−j|)` | `lru_cache` |
| 4 | Một mảng `buf` cho `left`, tính `right` gộp vào khi quét ngược | Giữ biến chạy thay cho mảng `right` | `ndp` ghi đè `buf` |
| 5 | Thay `− 1` bằng `− k` | `k = 0`: max hàng trên cộng vào | Kiểm bằng vét cạn |

## 19. Đáp án tham khảo

In [23]:
r1 = [Solution().maxPoints(g) for g in ([[3]], [[1, 2], [3, 4]], [[5, 0], [0, 5]])]
print("Bài 1:", r1); assert r1 == [3, 6, 9]
print("Bài 2:"); assert max_points_scan_demo([[0, 0, 10], [10, 0, 0]]) == 18

Bài 1: [3, 6, 9]
Bài 2:
hàng 0: [0, 0, 10]
hàng 1: L=[0, 0, 10], R=[8, 9, 10] -> dp=[18, 9, 10]


In [24]:
from functools import lru_cache

def max_points_topdown(points):
    m, n = len(points), len(points[0])
    @lru_cache(maxsize=None)
    def f(r, c):
        if r == m - 1:
            return points[r][c]
        return points[r][c] + max(f(r + 1, j) - abs(c - j) for j in range(n))
    res = max(f(0, c) for c in range(n))
    f.cache_clear()
    return res

def max_points_one_buffer(points):
    n = len(points[0])
    dp = points[0][:]
    buf = [0] * n
    for row in points[1:]:
        best = -1
        for c in range(n):                         # buf = left
            best = max(best - 1, dp[c]); buf[c] = best
        best = -1
        for c in range(n - 1, -1, -1):             # gộp right ngay khi quét ngược
            best = max(best - 1, dp[c])
            buf[c] = row[c] + max(buf[c], best)
        dp, buf = buf, dp
    return max(dp)

rng = random.Random(3)
for _ in range(300):
    g = [[rng.randint(0, 30) for _ in range(rng.randint(1, 6))]]
    g += [[rng.randint(0, 30) for _ in range(len(g[0]))] for _ in range(rng.randint(0, 5))]
    want = Solution().maxPoints(g)
    assert max_points_topdown(g) == want == max_points_one_buffer(g)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [25]:
def max_points_k(points, k):
    n = len(points[0]); dp = points[0][:]
    for row in points[1:]:
        left = dp[:]
        for c in range(1, n): left[c] = max(left[c], left[c - 1] - k)
        right = dp[:]
        for c in range(n - 2, -1, -1): right[c] = max(right[c], right[c + 1] - k)
        dp = [row[c] + max(left[c], right[c]) for c in range(n)]
    return max(dp)

def max_points_k_brute(points, k):
    m, n = len(points), len(points[0])
    return max(sum(points[r][c] for r, c in enumerate(cols)) - k * sum(abs(a - b) for a, b in zip(cols, cols[1:]))
               for cols in product(range(n), repeat=m))

rng = random.Random(9)
for _ in range(400):
    m, n, k = rng.randint(1, 4), rng.randint(1, 5), rng.randint(0, 4)
    g = [[rng.randint(0, 15) for _ in range(n)] for _ in range(m)]
    assert max_points_k(g, k) == max_points_k_brute(g, k), (g, k)
    if k == 1:
        assert max_points_k(g, 1) == Solution().maxPoints(g)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", max_points_k(examples[0][0], 0), max_points_k(examples[0][0], 3))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 11 8


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
dp = points[0][:]
for row in points[1:]:
    L = dp[:]
    for c in range(1, n): L[c] = max(L[c], L[c-1] - 1)
    R = dp[:]
    for c in range(n-2, -1, -1): R[c] = max(R[c], R[c+1] - 1)
    dp = [row[c] + max(L[c], R[c]) for c in range(n)]
return max(dp)
```

**Checklist:**
- [ ] Quét **cả hai** chiều (trái và phải)?
- [ ] Mỗi bước quét **trừ 1** (chi phí đổi cột)?
- [ ] Hàng đầu: `dp = points[0]` (không trừ gì)?
- [ ] `O(m·n)`, không có vòng `O(n²)` mỗi hàng?
- [ ] Trả về `max(dp)` của hàng cuối.